In [ ]:
import sys
import os
import numpy as np
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.utils import read_image_rgb, show_image

import matplotlib.pyplot as plt
%matplotlib inline

print("Setup complete. Project root set to:", project_root)

In [ ]:
input_path = os.path.join(project_root, 'data', 'input', 'test.jpg')

try:
    original_image = read_image_rgb(input_path)
    print(f"Image loaded successfully. Shape: {original_image.shape}")

except Exception as e:
    print(f"Failed to load image: {e}")

In [ ]:
import cv2

def compute_energy_map(img):
    """
    Computes the energy map of an image using the Dual-Gradient Energy function.
    
    The energy of a pixel is defined as E(p) = |dI/dx| + |dI/dy|.
    We use the Sobel operator to approximate the gradients.
    
    Args:
        img: Input image (RGB numpy array).
    
    Returns:
        energy_map: 2D numpy array (float64) of the same height and width as input.
    """

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    
    dx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    dy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    
    energy_map = np.abs(dx) + np.abs(dy)
    
    return energy_map

energy_map = compute_energy_map(original_image)

In [ ]:
fig, (ax_original, ax_energy) = plt.subplots(1, 2, figsize=(15, 8))

# Original
ax_original.imshow(original_image)
ax_original.set_title("Original Image")
ax_original.axis('off')

# Energy 
energy_heatmap = ax_energy.imshow(energy_map, cmap='inferno')
ax_energy.set_title("Energy Map (Gradient Magnitude)")
ax_energy.axis('off')

fig.colorbar(energy_heatmap, ax=ax_energy, fraction=0.046, pad=0.04)

plt.show()

In [ ]:
from numba import jit

@jit(nopython=True)
def compute_cumulative_map(energy):
    """
    Computes the cumulative minimum energy map using Dynamic Programming.
    
    Args:
        energy: The energy map.
        
    Returns:
        M: The cumulative cost matrix.
    """
    rows, cols = energy.shape
    
    # Create a copy to store the cumulative costs (M)
    M = energy.copy()

    # Iterate from the second row down to the last row
    for r in range(1, rows):
        for c in range(cols):
            # Logic: Find the minimum of the 3 upper neighbors
            
            # Handle left edge case (Column 0)
            if c == 0:
                min_prev = min(M[r-1, c], M[r-1, c+1])
            
            # Handle right edge case (Last Column)
            elif c == cols - 1:
                min_prev = min(M[r-1, c-1], M[r-1, c])
            
            # Standard case (Middle pixels)
            else:
                min_prev = min(M[r-1, c-1], M[r-1, c], M[r-1, c+1])
            
            # Add the minimum path cost to the current pixel's energy
            M[r, c] += min_prev
            
    return M

cumulative_map = compute_cumulative_map(energy_map)
print(f"Cumulative Map computed. Shape: {cumulative_map.shape}")

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(cumulative_map, cmap='viridis')
# Lighter = Higher Cumulative Cost  
plt.title("Cumulative Cost Matrix (M)")
plt.colorbar(label="Total Path Cost")
plt.axis('off')
plt.show()